In [ ]:
%load_ext autoreload
%autoreload 2

<h1 style="text-align: center;">Convergence and the CFL Condition</h1>

As seen in the previous steps, the values setted in the numerical parameters have an influence in the result we get. Then so, in this part we are going to see what happens whe the size of the grid increases:

$$
u(x,0) =
\begin{cases}
1, & x \in [0,0.5) \cup (1,2] \\
2, & x \in [0.5,1]
\end{cases}
$$
$$
u_i^{n+1} = u_i^n - c \frac{\Delta t}{\Delta x} (u_i^n - u_{i-1}^n)
$$

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline

def linearconvergence(nx: int, nt:int =20, dt:float =0.025, c:float =1):
    dx = 2/(nx-1)

    u = np.ones(nx) # u_i^(n+1)
    u[int(.5/dx):int(1/dx + 1)] = 2

    u_n = np.ones(nx)

    for n in range(nt):
        u_n = u.copy()
        for i in range(1, nx):
            u[i] = u_n[i] - c * (dt/dx) * (u_n[i] - u_n[i-1])
    plt.figure()
    plt.plot(np.linspace(0, 2, nx), u)
    return 1

 - Once the function has been defined let's try some values for nx and see how the output changes.

In [ ]:
u1 = linearconvergence(nx=41)
u2 = linearconvergence(nx=61)
u3 = linearconvergence(nx=71)
u4 = linearconvergence(nx=81)
u4 = linearconvergence(nx=85)


### Results

For small `nx` (41, 61, 71), the wave travels only a fraction of a cell per time step ($c\,\Delta t < \Delta x$). Each new value is then a weighted average of the cell and its left neighbour, so the solution stays between 1 and 2 but gets smeared out: this is **numerical diffusion**, an artifact of the scheme, not part of the physics. As `nx` increases, that fraction gets closer to one full cell, the diffusion decreases, and the wave looks more square.

However, $c\,\Delta t$ is constant ($c = 1$, $\Delta t = 0.025$) while $\Delta x = 2/(nx-1)$ keeps shrinking. For `nx = 85`, $\Delta x \approx 0.0238 < c\,\Delta t = 0.025$, so in one time step the wave travels more than one cell. But the update `u[i] = un[i] - c*dt/dx*(un[i] - un[i-1])` only uses `un[i]` and `un[i-1]`, so information can move at most one cell per step. The scheme can no longer keep up with the wave: it produces values outside $[1, 2]$, these are fed into the next step and amplified, and the solution blows up.

Stability therefore requires the Courant number<sup>1</sup> $\sigma = \dfrac{c\,\Delta t}{\Delta x}$ to satisfy $\sigma \le 1$, i.e. $\Delta t \le \Delta x / c$. For `nx = 85`, this gives $\Delta t_{max} \approx 0.0238$.
_________________________________________________________________

<sup>1</sup>The Courant number is the ratio between the distance the wave travels in one time step ($c\,\Delta t$) and the grid spacing ($\Delta x$); in other words, the number of cells the physical information crosses per time step. Since the upwind scheme can only pass information from one cell to its neighbour per step, it can only follow the wave if $\sigma \le 1$. This is the CFL condition (Courant, Friedrichs and Lewy, 1928). Besides stability, $\sigma$ also affects accuracy: for this scheme, the closer $\sigma$ is to 1, the less numerical diffusion.

- Then so, let's implement the Courant number into the code:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def linearconvergence_CFL(nx:int, nt:int=20, c:float=1, sigma:float=.5):
    dx = 2/(nx-1)
    dt = (sigma * dx)/c

    u = np.ones(nx) # u_i^(n+1)
    u[int(.5/dx):int(1/dx + 1)] = 2

    u_n = np.ones(nx)

    for n in range(nt):
        u_n = u.copy()
        for i in range(1, nx):
            u[i] = u_n[i] - c * (dt/dx) * (u_n[i] - u_n[i-1])
    plt.figure()
    plt.plot(np.linspace(0, 2, nx), u)
    return 1


- Now let's see what the result is when the CFL condition is applied:

In [ ]:
linearconvergence_CFL(nx=41)
linearconvergence_CFL(nx=61)
linearconvergence_CFL(nx=81)
linearconvergence_CFL(nx=101)
linearconvergence_CFL(nx=121)